In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from pathlib import Path
from sklearn.model_selection import train_test_split
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from sklearn.model_selection import KFold
from chemprop import data, featurizers, models, nn
from rdkit.Chem import MolFromSmiles
import concurrent.futures
import logging
from chemprop.data import datapoints, dataloader, MoleculeDataset
from chemprop.featurizers import SimpleMoleculeMolGraphFeaturizer
from chemprop.data.datapoints import MoleculeDatapoint
import torch
import chemprop.nn.metrics as chem_metrics
from chemprop.nn.agg import (
    MultiHeadAttentiveAggregation,
    GatedAttentiveAggregation,
    GatedAttentiveAggregationv2
)
from assets.functionchem import *
from assets.chempropcombination import *

df = pd.read_csv("./data/lipodata.csv")

df = df.rename(columns={"target": "LogD"})

df["scaffold"] = df["smiles"].apply(compute_scaffold)
scaffold_list = df["scaffold"].unique()
# Step 2: Split the list of scaffolds into train, test, and validation
train_scaffolds, temp_scaffolds = train_test_split(scaffold_list, test_size=0.2, random_state=42)
test_scaffolds, val_scaffolds = train_test_split(temp_scaffolds, test_size=0.5, random_state=42)

# Step 3: Create the train, test, and validation sets by filtering the original DataFrame based on scaffold
train_df = df[df['scaffold'].isin(train_scaffolds)]
test_df = df[df['scaffold'].isin(test_scaffolds)]
val_df = df[df['scaffold'].isin(val_scaffolds)]

# Step 4: Verify the distribution of scaffolds in each set
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Validation set size: {len(val_df)}")

results_df = run_chemprop_mp_agg_benchmark(
    train_df,
    val_df,
    test_df,
    target_column="LogD",
    max_epochs=200
)

Seed set to 42


Training set size: 7391
Test set size: 924
Validation set size: 924


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/tanush/anaconda3/envs/chemprop/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/tanush/Documents/WB_PBPK_MODEL_FINALIZED/submissionGit18MARCH_FinalizedCode/checkpoint/checkpoints_variants/LogD/BondMP3_Mean exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.



Training: BondMP3 + Mean


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

...

Output()

`Trainer.fit` stopped: `max_epochs=200` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Test: {'Target': 'LogD', 'MessagePassing': 'AtomMP6', 'Aggregation': 'MultiHead12', 'Test_MAE': 0.2931774854660034, 'Test_RMSE': 0.4098604917526245, 'Test_R2': 0.8754772543907166}

Final comparison:
   Target MessagePassing     Aggregation  Test_MAE  Test_RMSE   Test_R2
8    LogD        BondMP4      Attentive1  0.277402   0.393949  0.884958
20   LogD        BondMP6      Attentive1  0.278740   0.395608  0.883987
12   LogD        BondMP5            Mean  0.279673   0.395373  0.884125
23   LogD        BondMP6     MultiHead12  0.281506   0.402500  0.879910
15   LogD        BondMP5      MultiHead4  0.281587   0.402805  0.879727
21   LogD        BondMP6      MultiHead4  0.284648   0.408906  0.876056
22   LogD        BondMP6      MultiHead8  0.285475   0.404597  0.878655
19   LogD        BondMP6  GatedAttentive  0.286564   0.411163  0.874684
17   LogD        BondMP5     MultiHead12  0.286797   0.406124  0.877737
13   LogD        BondMP5  GatedAttentive  0.287678   0.407076  0.877164
14   LogD